# Diabetes Prediction — Exploratory Data Analysis

This notebook explores the diabetes prediction dataset before model training. The goal is to understand the dataset structure, check data quality, examine class imbalance, and look at how the model features relate to the target.

The EDA is intentionally kept separate from model training so that the final training pipeline can handle preprocessing consistently and without data leakage.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


# Make the notebook work whether it is opened from the
# project root or directly from the notebooks/ folder.
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "diabetes_prediction_dataset.csv"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")

## 1. Load the Dataset

The dataset is restored through DVC and stored locally at `data/diabetes_prediction_dataset.csv`.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

df.head()

## 2. Dataset Structure

The model uses eight input features and one binary target:

- `gender`
- `age`
- `hypertension`
- `heart_disease`
- `smoking_history`
- `bmi`
- `HbA1c_level`
- `blood_glucose_level`
- Target: `diabetes`

In [ ]:
df.info()

In [ ]:
df.dtypes.to_frame("dtype")

## 3. Data Quality Checks

Before analyzing distributions, I check for missing values and duplicate rows. Duplicate rows are removed before the modeling dataset is used.

In [ ]:
quality_summary = pd.DataFrame(
    {
        "missing_values": df.isna().sum(),
        "missing_percent": (
            df.isna().mean() * 100
        ).round(2),
        "unique_values": df.nunique(),
    }
)

quality_summary

In [ ]:
duplicate_count = int(
    df.duplicated().sum()
)

print(
    f"Duplicate rows: {duplicate_count:,}"
)

print(
    f"Duplicate percentage: "
    f"{duplicate_count / len(df) * 100:.2f}%"
)

In [ ]:
clean_df = (
    df
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows before duplicate removal: "
    f"{len(df):,}"
)

print(
    f"Rows after duplicate removal: "
    f"{len(clean_df):,}"
)

print(
    f"Rows removed: "
    f"{len(df) - len(clean_df):,}"
)

## 4. Target Distribution

The `diabetes` column is the binary target. Because diabetes cases are less common than non-diabetes cases, accuracy alone would not be a strong enough evaluation metric.

This is why the modeling portion of the project also evaluates precision, recall, F1, F-beta, AUC-ROC, and Average Precision.

In [ ]:
target_counts = (
    clean_df["diabetes"]
    .value_counts()
    .sort_index()
)

target_percent = (
    clean_df["diabetes"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

target_summary = pd.DataFrame(
    {
        "count": target_counts,
        "percent": target_percent.round(2),
    }
)

target_summary.index = [
    "No Diabetes (0)",
    "Diabetes (1)",
]

target_summary

In [ ]:
fig, ax = plt.subplots(
    figsize=(6, 4)
)

target_summary["count"].plot(
    kind="bar",
    ax=ax,
)

ax.set_title(
    "Diabetes Target Distribution"
)
ax.set_xlabel(
    "Target Class"
)
ax.set_ylabel(
    "Number of Records"
)
ax.tick_params(
    axis="x",
    rotation=0,
)

plt.tight_layout()
plt.show()

## 5. Numeric Feature Summary

The main continuous numeric features are age, BMI, HbA1c level, and blood glucose level.

In [ ]:
numeric_features = [
    "age",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
]

clean_df[
    numeric_features
].describe().T

In [ ]:
for feature in numeric_features:
    fig, ax = plt.subplots(
        figsize=(7, 4)
    )

    clean_df[feature].plot(
        kind="hist",
        bins=30,
        ax=ax,
    )

    ax.set_title(
        f"Distribution of {feature}"
    )
    ax.set_xlabel(
        feature
    )
    ax.set_ylabel(
        "Frequency"
    )

    plt.tight_layout()
    plt.show()

## 6. Numeric Features by Diabetes Status

These summaries compare each numeric feature between the two target classes. This does not establish medical causation; it only shows how the values are distributed in this dataset.

In [ ]:
numeric_by_target = (
    clean_df
    .groupby("diabetes")[
        numeric_features
    ]
    .agg(
        ["mean", "median"]
    )
    .round(2)
)

numeric_by_target

In [ ]:
for feature in numeric_features:
    grouped = [
        clean_df.loc[
            clean_df["diabetes"] == target,
            feature,
        ]
        for target in [0, 1]
    ]

    fig, ax = plt.subplots(
        figsize=(6, 4)
    )

    ax.boxplot(
        grouped,
        tick_labels=[
            "No Diabetes",
            "Diabetes",
        ],
    )

    ax.set_title(
        f"{feature} by Diabetes Status"
    )
    ax.set_ylabel(
        feature
    )

    plt.tight_layout()
    plt.show()

## 7. Categorical Feature Distributions

The two categorical features are `gender` and `smoking_history`. These are one-hot encoded inside the training pipeline.

In [ ]:
categorical_features = [
    "gender",
    "smoking_history",
]

for feature in categorical_features:
    summary = pd.DataFrame(
        {
            "count": (
                clean_df[feature]
                .value_counts()
            ),
            "percent": (
                clean_df[feature]
                .value_counts(
                    normalize=True
                )
                * 100
            ).round(2),
        }
    )

    print(
        f"\n{feature}"
    )

    display(summary)

In [ ]:
for feature in categorical_features:
    counts = (
        clean_df[feature]
        .value_counts()
    )

    fig, ax = plt.subplots(
        figsize=(7, 4)
    )

    counts.plot(
        kind="bar",
        ax=ax,
    )

    ax.set_title(
        f"Distribution of {feature}"
    )
    ax.set_xlabel(
        feature
    )
    ax.set_ylabel(
        "Count"
    )
    ax.tick_params(
        axis="x",
        rotation=45,
    )

    plt.tight_layout()
    plt.show()

## 8. Binary Medical-History Features

`hypertension` and `heart_disease` are already represented as binary 0/1 features.

In [ ]:
binary_features = [
    "hypertension",
    "heart_disease",
]

binary_summary = pd.DataFrame(
    {
        feature: {
            "No (0)": int(
                (clean_df[feature] == 0)
                .sum()
            ),
            "Yes (1)": int(
                (clean_df[feature] == 1)
                .sum()
            ),
            "Positive Rate (%)": round(
                clean_df[feature].mean()
                * 100,
                2,
            ),
        }
        for feature in binary_features
    }
).T

binary_summary

## 9. Diabetes Rate Across Categorical and Binary Features

The tables below make it easier to see whether the observed diabetes rate differs across categories in this dataset.

In [ ]:
for feature in [
    "gender",
    "smoking_history",
    "hypertension",
    "heart_disease",
]:
    summary = (
        clean_df
        .groupby(feature)["diabetes"]
        .agg(
            records="count",
            diabetes_rate="mean",
        )
    )

    summary[
        "diabetes_rate"
    ] = (
        summary[
            "diabetes_rate"
        ]
        * 100
    ).round(2)

    print(
        f"\nDiabetes rate by {feature}"
    )

    display(summary)

## 10. Correlation Between Numeric Features

Correlation is only a simple linear relationship measure, but it can still be useful for identifying features that move together.

Categorical features are not included in this matrix because they are encoded later in the model pipeline.

In [ ]:
correlation_columns = [
    "age",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
    "hypertension",
    "heart_disease",
    "diabetes",
]

correlation_matrix = (
    clean_df[
        correlation_columns
    ]
    .corr()
    .round(2)
)

correlation_matrix

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 6)
)

image = ax.imshow(
    correlation_matrix,
    aspect="auto",
)

ax.set_xticks(
    range(
        len(
            correlation_matrix.columns
        )
    )
)

ax.set_xticklabels(
    correlation_matrix.columns,
    rotation=45,
    ha="right",
)

ax.set_yticks(
    range(
        len(
            correlation_matrix.index
        )
    )
)

ax.set_yticklabels(
    correlation_matrix.index
)

for row in range(
    len(
        correlation_matrix.index
    )
):
    for column in range(
        len(
            correlation_matrix.columns
        )
    ):
        ax.text(
            column,
            row,
            correlation_matrix.iloc[
                row,
                column,
            ],
            ha="center",
            va="center",
        )

ax.set_title(
    "Correlation Matrix"
)

fig.colorbar(
    image,
    ax=ax,
)

plt.tight_layout()
plt.show()

## 11. EDA Summary

The main findings from this exploration are:

- The raw dataset contains about 100,000 records.
- Duplicate observations are present, so duplicates are removed before model development.
- After duplicate removal, the modeling dataset contains roughly 96,000 unique observations.
- The diabetes target is imbalanced, with the positive class representing only a relatively small portion of the data.
- Because of that imbalance, model selection should not rely on accuracy alone.
- The project evaluates precision, recall, F1, F-beta, AUC-ROC, and Average Precision in addition to accuracy.
- Numeric features are scaled inside the Scikit-Learn pipeline.
- Categorical features are encoded inside the pipeline.
- Preprocessing is fitted only on the training data to help prevent data leakage.

This EDA is descriptive. Relationships shown here should not be interpreted as medical causation or clinical guidance.